# Thermal Resistance Estimate and ANSYS Validation

This notebook provides the analytical thermal-resistance check used to validate the baseline ANSYS thermal model.

Its purpose is to:

- document the frozen thermal inputs,
- estimate the baseline junction temperature analytically,
- compare the analytical estimate with ANSYS,
- quantify the no-heat-sink reference case, and
- convert ANSYS case/package temperatures into estimated junction temperatures for the 5 A, 10 A, 20 A, 10 W and 15 W thermal cases.

The analytical network is a validation tool. ANSYS remains the detailed thermal solver used for final material and geometry comparisons.


## 1. Frozen Thermal Inputs

| Parameter | Value |
|---|---:|
| Ambient temperature | 25°C |
| Baseline MOSFET heat load | 1.505 W |
| Natural convection coefficient | 10 W/m²K |
| Baseline exposed convection area | 0.0533 m² |
| MOSFET RθJC | 1.5°C/W |
| No-heat-sink RθJA | 62°C/W |
| TIM | TGP5000 |
| TIM thickness | 1.5 mm |
| TIM conductivity | 5 W/mK |
| Heat-source dimensions | 15.8 mm × 10.0 mm |
| Project junction-temperature target | 125°C |
| IRFZ44N absolute junction-temperature limit | 175°C |

The 125°C value is a **project-defined design target** that retains a 50°C margin below the IRFZ44N 175°C absolute maximum; it is not a manufacturer-recommended continuous operating temperature.


## 2. Thermal Path and Baseline Assumptions

The cooled thermal path is:

`MOSFET junction → case/tab → TIM → heat-sink base/fins → ambient air`

The simplified analytical model represents this path as a one-dimensional series resistance network. Perfect contact is assumed between the MOSFET case, TIM and heat sink, while radiation, lead conduction and detailed three-dimensional heat spreading are neglected.


The baseline convection area of **0.0533 m²** is the area retained from the documented ANSYS baseline model for the analytical validation. Geometry-derived surface-area estimates used later for the separate convection sensitivity are identified explicitly and are not substituted into this baseline validation.

## 3. Thermal-Resistance Path


The corresponding total thermal resistance is:

$$
R_{\theta,\mathrm{total}}
=
R_{\theta JC}
+
R_{\theta TIM}
+
R_{\theta HS}
+
R_{\theta conv}
$$

where:

- $R_{\theta JC}$ is the junction-to-case thermal resistance of the MOSFET.
- $R_{\theta TIM}$ is the thermal resistance through the thermal interface material.
- $R_{\theta HS}$ is the conduction resistance through the heat-sink base and into the fins.
- $R_{\theta conv}$ is the convection resistance from the exposed heat-sink surface to the surrounding air.

For this first simplified calculation, perfect contact is assumed between the MOSFET case, TIM and heat sink. Therefore, separate contact resistances at the case–TIM and TIM–heat-sink interfaces are neglected.

The heat-sink fins are included through the total exposed surface area used in the convection calculation. A separate fin resistance is not added in this initial model.

## 4. No-Heat-Sink Reference

The no-heat-sink case is evaluated separately using the datasheet-style junction-to-ambient resistance:

`Tj = Ta + Ploss × RθJA`

This reference is used only to quantify the importance of dedicated cooling and is not treated as equivalent to the detailed ANSYS geometry.


In [1]:
import pandas as pd

# No-heat-sink reference calculation

ambient_temperature = 25.0       # °C
heat_input = 1.505               # W
r_theta_ja = 62.0                # °C/W 

maximum_junction_temperature = 175.0   # °C
target_junction_temperature = 125.0  # °C - project-defined design target

tj_no_heatsink = (
    ambient_temperature
    + heat_input * r_theta_ja
)

margin_to_maximum = (
    maximum_junction_temperature
    - tj_no_heatsink
)

margin_to_target = (
    target_junction_temperature
    - tj_no_heatsink
)

no_heatsink_results = pd.DataFrame({
    "Case": ["No heat sink"],
    "Heat input (W)": [heat_input],
    "Thermal resistance (°C/W)": [r_theta_ja],
    "Estimated junction temperature (°C)": [tj_no_heatsink],
    "Margin below maximum (°C)": [margin_to_maximum],
    "Margin below 125°C project target (°C)": [margin_to_target]
})

no_heatsink_results.round(2)

,Case,Heat input (W),Thermal resistance (°C/W),Estimated junction temperature (°C),Margin below maximum (°C),Margin below 125°C project target (°C)
0,No heat sink,1.5,62.0,118.31,56.69,6.69


## 5. Analytical Baseline Heat-Sink Estimate


In [2]:
# Pre-FEA junction-temperature estimate
# Final comparison: aluminium and copper heat-sink constructions

import pandas as pd

# -------------------------------------------------
# Fixed electrical and MOSFET inputs
# -------------------------------------------------

P_loss = 1.505          # W, conservative baseline MOSFET heat load
T_ambient = 25.0        # degC
R_jc = 1.5              # degC/W, IRFZ44N junction-to-case resistance

# MOSFET/TIM contact area
source_width = 15.8e-3   # m
source_height = 10.0e-3  # m
A_contact = source_width * source_height

# -------------------------------------------------
# TIM: TGP 5000
# Fixed project values documented in data/material-properties.csv and docs/sources.md
# -------------------------------------------------

t_tim = 1.5e-3          # m
k_tim = 5.0             # W/(m K)

# -------------------------------------------------
# Heat-sink materials
# Fixed project values documented in data/material-properties.csv and docs/sources.md
# -------------------------------------------------

k_aluminium = 170.0     # W/(m K), aluminium 6061-T6
k_copper = 390.0        # W/(m K), C11000 copper

# -------------------------------------------------
# Heat-sink geometry and convection
# Frozen baseline heat-sink geometry and convection inputs
# -------------------------------------------------

t_base = 5.0e-3         # m, baseline heat-sink base thickness
h = 10.0                 # W/(m^2 K), natural convection coefficient
A_surface = 0.0533      # m^2, total exposed base and fin surface area

# -------------------------------------------------
# Common thermal resistances
# -------------------------------------------------

R_tim = t_tim / (k_tim * A_contact)
R_convection = 1.0 / (h * A_surface)

# -------------------------------------------------
# Function for calculating each construction
# -------------------------------------------------

def calculate_case(case_name, base_conductivity, fin_material):
    """
    Calculate the simplified pre-FEA thermal resistance and junction
    temperature for one heat-sink construction.

    In this simple model, the heat-sink conduction term represents
    conduction through the base. Fin material is recorded for reference,
    while the fins are represented through the exposed convection area.
    """

    R_base = t_base / (base_conductivity * A_contact)

    R_total = (
        R_jc
        + R_tim
        + R_base
        + R_convection
    )

    temperature_rise = P_loss * R_total
    T_junction = T_ambient + temperature_rise

    return {
        "Construction": case_name,
        "Base material": (
            "Aluminium 6061-T6"
            if base_conductivity == k_aluminium
            else "C11000 copper"
        ),
        "Fin material": fin_material,
        "R_jc (degC/W)": R_jc,
        "R_TIM (degC/W)": R_tim,
        "R_base (degC/W)": R_base,
        "R_convection (degC/W)": R_convection,
        "R_total (degC/W)": R_total,
        "Temperature rise (degC)": temperature_rise,
        "Estimated Tj (degC)": T_junction,
    }

# -------------------------------------------------
# No-heat-sink reference case
# -------------------------------------------------

R_ja = 62.0  # degC/W, IRFZ44N junction-to-ambient thermal resistance

temperature_rise_no_heatsink = P_loss * R_ja
T_junction_no_heatsink = T_ambient + temperature_rise_no_heatsink

no_heatsink_case = {
    "Construction": "No heat sink",
    "Base material": "None",
    "Fin material": "None",
    "R_jc (degC/W)": None,
    "R_TIM (degC/W)": None,
    "R_base (degC/W)": None,
    "R_convection (degC/W)": None,
    "R_total (degC/W)": R_ja,
    "Temperature rise (degC)": temperature_rise_no_heatsink,
    "Estimated Tj (degC)": T_junction_no_heatsink,
}

# -------------------------------------------------
# Define the three cooling constructions
# -------------------------------------------------

results = [
    no_heatsink_case,
    calculate_case(
        case_name="All aluminium",
        base_conductivity=k_aluminium,
        fin_material="Aluminium 6061-T6",
    ),
    calculate_case(
        case_name="All copper",
        base_conductivity=k_copper,
        fin_material="C11000 copper",
    ),
]

results_df = pd.DataFrame(results)

# Round numerical results for clearer display
numeric_columns = results_df.select_dtypes(include="number").columns
results_df[numeric_columns] = results_df[numeric_columns].round(3)

print(f"Contact area: {A_contact * 1e6:.1f} mm^2")
print(f"TGP 5000 TIM resistance: {R_tim:.3f} degC/W")
print(f"Convection resistance: {R_convection:.3f} degC/W")
print()

display(results_df)

Contact area: 158.0 mm^2
TGP 5000 TIM resistance: 1.899 degC/W
Convection resistance: 1.876 degC/W



,Construction,Base material,Fin material,R_jc (degC/W),R_TIM (degC/W),R_base (degC/W),R_convection (degC/W),R_total (degC/W),Temperature rise (degC),Estimated Tj (degC)
0,No heat sink,None,None,NaN,NaN,NaN,NaN,62.000,93.310,118.310
1,All aluminium,Aluminium 6061-T6,Aluminium 6061-T6,1.5,1.899,0.186,1.876,5.461,8.219,33.219
2,All copper,C11000 copper,C11000 copper,1.5,1.899,0.081,1.876,5.356,8.061,33.061


## 6. Analytical vs ANSYS Baseline Validation

For the 1.505 W aluminium baseline case:

- ANSYS maximum case/package temperature ≈ **30.63°C**
- ANSYS-derived junction temperature ≈ **32.88°C**
- Analytical junction-temperature estimate ≈ **33.22°C**
- Absolute difference ≈ **0.34°C**
- Difference normalised to the ANSYS temperature rise above 25°C ambient ≈ **4.2%**

The analytical and FEA results therefore agree within the project's practical **10% temperature-rise validation guide**. The percentage is referenced to **temperature rise above ambient**, rather than to the Celsius temperature itself, because Celsius has an arbitrary zero point.

| Validation quantity | Value |
|---|---:|
| Analytical junction-temperature estimate | 33.22°C |
| ANSYS maximum case/package temperature | 30.63°C |
| ANSYS-derived junction-temperature estimate | 32.88°C |
| Absolute junction-temperature difference | 0.34°C |
| Difference / ANSYS temperature rise above ambient | 4.2% |

## 7. Heat-Input Consistency

The analytical model and ANSYS use the same baseline MOSFET heat input of **1.505 W**.

In ANSYS, the heat input is applied either as a total heat load or as the equivalent heat flux over the **15.8 mm × 10.0 mm** source face. Only one heat-input method is used at a time to avoid double counting.


## 8. Thermal Heat-Load Cases

### Electrically Derived Cases

| Electrical operating point | Conservative heat input |
|---|---:|
| 5 A | 0.534 W |
| 10 A | 1.505 W |
| 20 A | 4.760 W |

### Imposed Thermal-Stress Cases

| Thermal-stress case | Heat input |
|---|---:|
| High thermal stress | 10 W |
| Extreme thermal stress | 15 W |

The **10 W and 15 W** cases are imposed thermal-stress loads and are not presented as equivalent converter-current operating points.


## 9. Junction-Temperature Calculation from ANSYS

ANSYS provides the maximum temperature at the simplified MOSFET case / heat-source region.

The corresponding junction temperature is estimated using:

`Tj = Tc + P × RθJC`

where **RθJC = 1.5°C/W**.


In [3]:
import pandas as pd

power = [0.534, 1.505, 4.760, 10.0, 15.0]
current = ["5 A", "10 A", "20 A", "Stress case", "Stress case"]
aluminium_tc = [26.997, 30.627, 42.798, 62.39, 81.095]
copper_tc = [26.927, 30.43, 42.175, 61.082, 79.124]


R_theta_JC = 1.5  # °C/W

aluminium_tj = [
    tc + p * R_theta_JC
    for tc, p in zip(aluminium_tc, power)
]

copper_tj = [
    tc + p * R_theta_JC
    for tc, p in zip(copper_tc, power)
]

results = pd.DataFrame({
    "Power (W)": power,
    "Current": current,
    "Aluminium Tc (°C)": aluminium_tc,
    "Aluminium Tj (°C)": aluminium_tj,
    "Copper Tc (°C)": copper_tc,
    "Copper Tj (°C)": copper_tj
})

results.round(3)



,Power (W),Current,Aluminium Tc (°C),Aluminium Tj (°C),Copper Tc (°C),Copper Tj (°C)
0,0.534,5 A,26.997,27.798,26.927,27.728
1,1.505,10 A,30.627,32.884,30.430,32.688
2,4.760,20 A,42.798,49.938,42.175,49.315
3,10.000,Stress case,62.390,77.390,61.082,76.082
4,15.000,Stress case,81.095,103.595,79.124,101.624


### Interpretation

The junction temperature increases with MOSFET power dissipation for both
heatsink materials. Copper produces slightly lower temperatures than aluminium,
although the difference remains relatively small.

At 15 W, the calculated junction temperatures are approximately 103.6 °C for
aluminium and 101.6 °C for copper, corresponding to a reduction of around
2 °C when copper is used.

This suggests that while heatsink thermal conductivity affects the result,
other thermal resistances, particularly convection from the heatsink to the
surrounding air, have a significant influence on the overall thermal performance.

## 10. Validation Role and Limitations

The analytical thermal-resistance model is retained as an independent first-order validation check of the ANSYS model.

Key limitations are:

- one-dimensional heat-flow assumption,
- perfect thermal contacts,
- no explicit radiation term,
- no MOSFET-lead heat loss,
- simplified heat-sink conduction representation, and
- convection represented by a fixed natural-convection coefficient.

ANSYS is therefore used for the final material and geometry studies, while this notebook documents the analytical validation and junction-temperature conversion method.
